# Iris PCA (Python)

-----

## Setup

In [ ]:
# load-packages
# Import pandas for data frames
import pandas as pd
# Import numpy for numerical operations
import numpy as np
# Import matplotlib for plotting
import matplotlib.pyplot as plt
# Import seaborn for the scatter plot
import seaborn as sns
# Import the iris dataset loader from scikit-learn
from sklearn.datasets import load_iris
# Import the PCA class from scikit-learn
from sklearn.decomposition import PCA

In [ ]:
# load-data
# Load the iris dataset with features as a pandas data frame
iris = load_iris(as_frame=True)
# Copy the feature data frame so we don't modify the original
iris_df = iris.data.copy()
# Rename columns to match the R version of iris
iris_df.columns = ["Sepal.Length", "Sepal.Width", "Petal.Length", "Petal.Width"]
# Add a Species column by converting integer codes to species names
iris_df["Species"] = pd.Categorical.from_codes(iris.target, iris.target_names)
# Print the first few rows
iris_df.head()

In [ ]:
# select-numeric
# Keep only the four numeric measurement columns (drop Species)
iris_num = iris_df.iloc[:, :4]
# Print the first few rows of the numeric values
iris_num.head()

## Run PCA

In [ ]:
# run-pca
# Center each column and scale to unit variance (ddof=1 matches R's prcomp)
iris_scaled = (iris_num - iris_num.mean()) / iris_num.std(ddof=1)
# Create a PCA object that keeps all four components
pca = PCA(n_components=4)
# Fit PCA and compute the scores (coordinates of each flower on each PC)
pca_scores = pca.fit_transform(iris_scaled)
# Compute standard deviation of each component from its variance
sdev = np.sqrt(pca.explained_variance_)
# Print the standard deviations of each component
print("Standard deviations:", sdev)
# Build a data frame of loadings (rotation) with variables as rows
loadings = pd.DataFrame(
    # Transpose components so rows are variables and columns are PCs
    pca.components_.T,
    # Label rows with the original variable names
    index=iris_num.columns,
    # Label columns PC1 through PC4
    columns=[f"PC{i}" for i in range(1, 5)],
)
# Display the loadings (signs of PCs may be flipped relative to R)
loadings

In [ ]:
# pca-summary
# Build a summary table like R's summary(prcomp)
pca_summary = pd.DataFrame(
    {
        # Standard deviation of each component
        "Standard deviation": sdev,
        # Proportion of variance explained by each component
        "Proportion of Variance": pca.explained_variance_ratio_,
        # Running total of the proportion of variance explained
        "Cumulative Proportion": np.cumsum(pca.explained_variance_ratio_),
    },
    # Label rows PC1 through PC4
    index=[f"PC{i}" for i in range(1, 5)],
)
# Display the summary table, rounded to 4 decimals
pca_summary.round(4)

## Scree Plot

In [ ]:
# build-scree-data
# Build a data frame for the scree plot
scree_df = pd.DataFrame(
    {
        # Component labels PC1 through PC4
        "PC": [f"PC{i}" for i in range(1, len(pca.explained_variance_ratio_) + 1)],
        # Variance explained by each component
        "Variance": pca.explained_variance_ratio_,
    }
)

In [ ]:
# scree-plot
# Create a figure and axes for the scree plot
fig, ax = plt.subplots()
# Draw one bar per principal component
ax.bar(scree_df["PC"], scree_df["Variance"])
# Label the x-axis
ax.set_xlabel("PC")
# Label the y-axis
ax.set_ylabel("Proportion of variance explained")
# Show the plot
plt.show()

## Sample scores

In [ ]:
# pc-scores-plot
# Combine PC scores (first two components) with the species labels
scores = pd.DataFrame(pca_scores[:, :2], columns=["PC1", "PC2"])
# Add the species labels to the scores data frame
scores["Species"] = iris_df["Species"].values

# Create a figure and axes for the scatter plot
fig, ax = plt.subplots()
# Plot samples in PC1 vs PC2 space, colored by species
sns.scatterplot(data=scores, x="PC1", y="PC2", hue="Species", ax=ax)
# Label the x-axis with percent variance explained
ax.set_xlabel(f"PC1 ({100 * var_explained[0]:.1f}%)")
# Label the y-axis with percent variance explained
ax.set_ylabel(f"PC2 ({100 * var_explained[1]:.1f}%)")
# Show the plot
plt.show()

In [ ]:
# inspect-loadings
# Inspect loadings to see which features drive PC1 and PC2
loadings[["PC1", "PC2"]].round(3)